In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Problem Statement
To detect the language which can enter by the user 

# 2. Data Gathering

In [2]:
data = pd.read_csv("Language Detection.csv")
data

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English
...,...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...,Kannada
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...,Kannada
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...,Kannada
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...,Kannada


In [3]:
# Rename columns if needed
data.columns = ["Text", "Language"]

print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (10337, 2)


,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English


# 3. Data Cleaning


In [4]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"\d+", "", text)       # remove numbers
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    return text

data["Text"] = data["Text"].apply(clean_text)


# 4. Prepare Features & Labels


In [5]:
x = data["Text"]
y = data["Language"]

print("Languages:", np.unique(y))


Languages: ['Arabic' 'Danish' 'Dutch' 'English' 'French' 'German' 'Greek' 'Hindi'
 'Italian' 'Kannada' 'Malayalam' 'Portugeese' 'Russian' 'Spanish'
 'Sweedish' 'Tamil' 'Turkish']


# 5. Convert Text → Features
IMPORTANT FIX: character n-grams


In [6]:
cv = CountVectorizer(analyzer='char', ngram_range=(2, 4))
X = cv.fit_transform(x)


# 6. Train-Test Split


In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Train Model


In [8]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


# 8. Evaluate Model


In [9]:
y_pred = model.predict(x_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))




Accuracy: 0.9738878143133463

Classification Report:

              precision    recall  f1-score   support

      Arabic       1.00      0.99      1.00       106
      Danish       0.85      0.95      0.90        73
       Dutch       0.96      0.95      0.96       111
     English       0.97      0.99      0.98       291
      French       0.99      0.98      0.98       219
      German       0.99      0.94      0.96        93
       Greek       1.00      0.99      0.99        68
       Hindi       1.00      1.00      1.00        10
     Italian       0.98      0.96      0.97       145
     Kannada       1.00      1.00      1.00        66
   Malayalam       0.98      0.99      0.98       121
  Portugeese       0.97      0.96      0.96       144
     Russian       1.00      0.99      1.00       136
     Spanish       0.96      0.96      0.96       160
    Sweedish       0.97      0.94      0.95       133
       Tamil       1.00      1.00      1.00        87
     Turkish       0.98   

# 9. Predict Custom Input


In [10]:
def predict_language(text):
    text = clean_text(text)
    vector = cv.transform([text])
    prediction = model.predict(vector)
    return prediction[0]



# 10. Test Examples


In [11]:
print("\n--- Sample Predictions ---")

print("English:", predict_language("This is a language detection system"))
print("Hindi:", predict_language("यह एक भाषा पहचान प्रणाली है"))
print("French:", predict_language("Ceci est un système de détection de langue"))
print("Spanish:", predict_language("Este es un sistema de detección de idioma"))
print("German:", predict_language("Dies ist ein Sprachenerkennungssystem"))



--- Sample Predictions ---
English: English
Hindi: Hindi
French: French
Spanish: Spanish
German: German


# 11. User Input

In [12]:

user_input = input("Enter text: ")

prediction = predict_language(user_input)
print(f'User Input  : {user_input}/n')
print(f"Predicted Language: {prediction}")

User Input  :  مرحباً، أنا سوميت تريباثي، وأنا من الهند/n
Predicted Language: Arabic


In [13]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(cv, open("vectorizer.pkl", "wb"))